# Bootstrap — Create Lakehouses + Deploy All Notebooks

Workspace: 

## Cell 2 — Crear Lakehouses (corre PRIMERO, solo una vez)
Crea  y  si no existen.

## Cell 3 — Deploy Notebooks (corre despues)
Sube/actualiza los 4 notebooks:
| Notebook | Lakehouse Target |
|---|---|
| Bronze_BIX_LAH_DF2 | Heritage_Bronze_LH |
| Bronze_OneSite_LAH_DF2 | Heritage_Bronze_LH |
| Silver_BIX_LAH_DF2 | Heritage_Silver_LH |
| Silver_OneSite_LAH_DF2 | Heritage_Silver_LH |

In [ ]:
# ============================================================
# CELL 2 — Create Lakehouses (Heritage_Bronze_LH, Heritage_Silver_LH)
# Run ONCE — skip if lakehouses already exist
# ============================================================
import json, urllib.request, urllib.error

WS_ID = "d445bd8c-ec46-44dd-96e1-ea6593a5b99b"
FABRIC_BASE = "https://api.fabric.microsoft.com/v1"

token   = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def list_lakehouses():
    url = f"{FABRIC_BASE}/workspaces/{WS_ID}/lakehouses"
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read()).get("value",[])

def create_lh(name, existing_names):
    if name in existing_names:
        print(f"  EXISTS : {name}")
        return
    url  = f"{FABRIC_BASE}/workspaces/{WS_ID}/lakehouses"
    body = json.dumps({"displayName": name, "type": "Lakehouse"}).encode()
    req  = urllib.request.Request(url, data=body, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            data = json.loads(r.read() or b"{}")
            print(f"  CREATED: {name} (id={data.get('id','?')})")
    except urllib.error.HTTPError as e:
        body_err = e.read()[:300]
        if e.code == 409:
            print(f"  EXISTS : {name} (409 conflict)")
        else:
            raise Exception(f"CREATE LH failed {e.code}: {body_err}")

existing_lh = list_lakehouses()
existing_names = {x["displayName"] for x in existing_lh}
print(f"Existing lakehouses ({len(existing_lh)}):")
for lh in existing_lh:
    print(f"  {lh['displayName']} ({lh['id']})")

print("
Creating lakehouses...")
create_lh("Heritage_Bronze_LH", existing_names)
create_lh("Heritage_Silver_LH", existing_names)

print("
Lakehouse step done.")

In [ ]:
# ============================================================
# Bootstrap — Deploy All Notebooks to Fabric
# Workspace: d445bd8c-ec46-44dd-96e1-ea6593a5b99b
# ============================================================
import base64, json, time
from notebookutils import mssparkutils

WS_ID = "d445bd8c-ec46-44dd-96e1-ea6593a5b99b"
FABRIC_BASE = "https://api.fabric.microsoft.com/v1"

NOTEBOOKS = [
    ("Bronze_BIX_LAH_DF2", r"""..."""),  # 29 KB
    ("Bronze_OneSite_LAH_DF2", r"""..."""),  # 112 KB
    ("Silver_BIX_LAH_DF2", r"""..."""),  # 27 KB
    ("Silver_OneSite_LAH_DF2", r"""..."""),  # 16 KB
]

def get_token():
    return mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

def get_headers(token):
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def list_notebooks(token):
    import urllib.request
    url = f"{FABRIC_BASE}/workspaces/{WS_ID}/items?type=Notebook"
    req = urllib.request.Request(url, headers=get_headers(token))
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["value"]

def poll_op(token, location, max_wait=120):
    import urllib.request
    deadline = time.time() + max_wait
    while time.time() < deadline:
        req = urllib.request.Request(location, headers=get_headers(token))
        with urllib.request.urlopen(req, timeout=30) as r:
            body = r.read()
            if not body.strip():
                return True
            data = json.loads(body)
            st = data.get("status","")
            if st in ("Succeeded","succeeded","Completed","completed",""):
                return True
            if st in ("Failed","failed"):
                raise Exception(f"Async op failed: {data}")
        time.sleep(4)
    raise TimeoutError("Async operation timed out")

def upsert_nb(token, display_name, b64_payload, existing):
    import urllib.request, urllib.error
    defn = {"format":"ipynb","parts":[{"path":"notebook-content.ipynb","payload":b64_payload,"payloadType":"InlineBase64"}]}

    found = next((x for x in existing if x.get("displayName")==display_name), None)

    if found:
        item_id = found["id"]
        url  = f"{FABRIC_BASE}/workspaces/{WS_ID}/items/{item_id}/updateDefinition"
        body = json.dumps({"definition":defn}).encode()
        req  = urllib.request.Request(url, data=body, headers=get_headers(token), method="POST")
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                code = r.status
                loc  = dict(r.headers).get("Location") or dict(r.headers).get("location")
                if code == 202 and loc:
                    poll_op(token, loc)
                print(f"  UPDATED: {display_name} ({item_id})")
                return item_id
        except urllib.error.HTTPError as e:
            raise Exception(f"UPDATE failed {e.code}: {e.read()[:300]}")
    else:
        url  = f"{FABRIC_BASE}/workspaces/{WS_ID}/items"
        body = json.dumps({"displayName":display_name,"type":"Notebook","definition":defn}).encode()
        req  = urllib.request.Request(url, data=body, headers=get_headers(token), method="POST")
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                code = r.status
                loc  = dict(r.headers).get("Location") or dict(r.headers).get("location")
                raw  = r.read()
                if code == 202 and loc:
                    poll_op(token, loc)
                    print(f"  CREATED: {display_name}")
                    return None
                data = json.loads(raw) if raw.strip() else {}
                item_id = data.get("id","?")
                print(f"  CREATED: {display_name} ({item_id})")
                return item_id
        except urllib.error.HTTPError as e:
            raise Exception(f"CREATE failed {e.code}: {e.read()[:300]}")

# ── MAIN ──────────────────────────────────────────────────────────
print("Getting token...")
token = get_token()
print("Token OK")

print(f"\nListing notebooks in workspace {WS_ID}...")
existing = list_notebooks(token)
print(f"Found {len(existing)} existing notebooks:")
for nb in existing:
    print(f"  {nb.get('displayName')} ({nb.get('id')})")

results = []
for name, payload in NOTEBOOKS:
    print(f"\n--- {name} ---")
    try:
        upsert_nb(token, name, payload, existing)
        results.append((name, "OK"))
    except Exception as e:
        print(f"  FAIL: {e}")
        results.append((name, str(e)[:200]))

print("\n" + "="*55)
print("BOOTSTRAP DEPLOY SUMMARY")
print("="*55)
ok_count = sum(1 for _,s in results if s=="OK")
for name, status in results:
    icon = "OK" if status=="OK" else "FAIL"
    print(f"  [{icon}] {name}" + (f": {status}" if status!="OK" else ""))
print(f"\n{ok_count}/{len(results)} notebooks deployed.")
if ok_count < len(results):
    raise Exception(f"{len(results)-ok_count} notebook(s) failed.")
print("Done.")